# **Step 1. Install and import libraries**

In [32]:

!pip install -q gradio scikit-learn pandas numpy matplotlib seaborn scipy


In [33]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")


Libraries imported successfully!


# **Step 2. Load the Breast Cancer Wisconsin dataset**

In [34]:

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset Shape:", X.shape)
print("\nFeature Names:")
print(X.columns.tolist())

print("\nTarget Classes:")
print(data.target_names)

print("\nFirst Five Records:")
display(X.head())


Dataset Shape: (569, 30)

Feature Names:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']

Target Classes:
['malignant' 'benign']

First Five Records:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# **Step 3. Dataset overview and preprocessing**

In [35]:

print("Missing Values:")
print(X.isnull().sum().sum())

print("\nDuplicate Records:")
print(X.duplicated().sum())

print("\nDataset Information:")
print(X.info())

print("\nStatistical Description:")
display(X.describe())

print("\nClass Distribution:")
print(y.value_counts())


Missing Values:
0

Duplicate Records:
0

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error        

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500



Class Distribution:
target
1    357
0    212
Name: count, dtype: int64


# **Visualize the class distribution.**

In [36]:

plt.figure(figsize=(6, 4))

sns.countplot(
    x=y.map({0: "Malignant", 1: "Benign"})
)

plt.title("Breast Cancer Class Distribution")
plt.xlabel("Cancer Type")
plt.ylabel("Number of Samples")
plt.show()


# **Step 4. Split the dataset**

In [37]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


Training Samples: 455
Testing Samples: 114


# **Step 5. Statistical feature analysis**
## Now we implement the three statistical techniques.

# **5.1 Variance based analysis**

Variance measures how much the values of a feature differ from their mean.

**Important methodological note:** the 30 measurements use different units and scales. Therefore, raw variance values should be interpreted carefully and should not be compared as if they were directly equivalent. In the final feature-selection pipeline, variance filtering is used only to remove constant features (variance = 0). This avoids discarding useful features merely because they are measured on a smaller numerical scale.

In [38]:
# Raw variance is shown for exploration only; it is scale-dependent.
variance_values = X_train.var()

variance_df = pd.DataFrame({
    "Feature": variance_values.index,
    "Raw Variance": variance_values.values
}).sort_values("Raw Variance", ascending=False)

display(variance_df)

ax = variance_df.head(15).sort_values("Raw Variance").plot(
    x="Feature", y="Raw Variance", kind="barh",
    figsize=(10, 6), legend=False
)
ax.set_title("Top 15 Features by Raw Variance (Scale-Dependent)")
ax.set_xlabel("Raw variance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

,Feature,Raw Variance
23,worst area,305728.169534
3,mean area,118986.752074
13,area error,1744.164596
22,worst perimeter,1101.911522
2,mean perimeter,583.185354
21,worst texture,38.756476
20,worst radius,22.753086
1,mean texture,19.406587
0,mean radius,12.245658
12,perimeter error,3.894602


# **5.2 Covariance and Correlation based analysis**
Covariance identifies how two features change in relation to each other.

In [39]:

covariance_matrix = X_train.cov()

plt.figure(figsize=(14, 10))

sns.heatmap(
    covariance_matrix,
    cmap="coolwarm",
    center=0,
    annot=False
)

plt.title("Feature Covariance Matrix")
plt.show()



# Correlation measures the strength and direction of the relationship between two features.

Values range from -1 to +1.
- +1: Perfect positive relationship.
- -1: Perfect negative relationship.
- 0: No linear relationship.

In [40]:

correlation_matrix = X_train.corr()

plt.figure(figsize=(16, 12))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    annot=False
)

plt.title("Feature Correlation Heatmap")
plt.show()


The following table identifies feature pairs with absolute correlation above 0.90. In the final pipeline, when two features are highly correlated, the feature with the stronger Fisher score is retained. This makes the redundancy decision more informed.

In [41]:

upper_triangle = correlation_matrix.abs().where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlation = (
    upper_triangle.stack()
    .reset_index()
)

high_correlation.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_correlation = high_correlation[
    high_correlation["Correlation"] > 0.90
]

display(high_correlation)


,Feature 1,Feature 2,Correlation
1,mean radius,mean perimeter,0.997789
2,mean radius,mean area,0.988886
19,mean radius,worst radius,0.970133
21,mean radius,worst perimeter,0.965936
22,mean radius,worst area,0.943351
48,mean texture,worst texture,0.912994
57,mean perimeter,mean area,0.987834
74,mean perimeter,worst radius,0.969674
76,mean perimeter,worst perimeter,0.970918
77,mean perimeter,worst area,0.943063


# **5.3 Class-separability-based selection**
# Fisher score calculation

Fisher score measures how well a feature separates the two target classes.

We calculate:
\[
F_j=\frac{\sum_c n_c(\mu_{jc}-\mu_j)^2}
{\sum_c n_c\sigma_{jc}^2}
\]

Where:
- \(n_c\) = number of samples in class c.
- \(\mu_{jc}\) = mean of feature j in class c.
- \(\mu_j\) = overall mean of feature j.
- \(\sigma_{jc}^2\) = within-class variance.

In [42]:

def calculate_fisher_score(X_data, y_data):

    scores = {}

    for feature in X_data.columns:

        overall_mean = X_data[feature].mean()

        between_class = 0
        within_class = 0

        for class_value in np.unique(y_data):

            values = X_data.loc[
                y_data == class_value, feature
            ]

            class_mean = values.mean()
            class_variance = values.var(ddof=0)

            between_class += (
                len(values) *
                (class_mean - overall_mean) ** 2
            )

            within_class += (
                len(values) * class_variance
            )

        scores[feature] = (
            between_class / (within_class + 1e-12)
        )

    return pd.Series(scores).sort_values(
        ascending=False
    )


fisher_scores = calculate_fisher_score(
    X_train,
    y_train
)

print("Top 15 Fisher Score Features:")
display(fisher_scores.head(15))

plt.figure(figsize=(10, 6))

fisher_scores.head(15).sort_values().plot(
    kind="barh"
)

plt.title("Top 15 Features Based on Fisher Score")
plt.xlabel("Fisher Score")
plt.show()


Top 15 Fisher Score Features:


,0
worst concave points,1.755460
worst perimeter,1.588518
mean concave points,1.552680
worst radius,1.517506
mean perimeter,1.248313
worst area,1.195836
mean radius,1.155881
mean area,1.036460
mean concavity,0.919150
worst concavity,0.807389


# **Step 6. Perform statistical feature selection**
We will combine all three techniques. The selector learns everything from the **training data only**.

Our selection strategy:
1. Remove zero-variance (constant) features.
2. Rank the remaining features by Fisher score.
3. Go down the ranking and drop any feature whose correlation with an already-kept, better-ranked feature is above 0.90.
4. Keep the top *k* remaining features (the preview below uses k = 15; Step 7 also tries k = 10 and 20).

The selector class is saved in its own file, `feature_selector.py`, so the trained model can be loaded later in any other script.

In [43]:
%%writefile feature_selector.py
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class StatisticalFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Training-fold feature selector:
    1. Remove constant features.
    2. Rank remaining features by Fisher score.
    3. Remove highly correlated features, keeping the higher Fisher-ranked feature.
    4. Keep the top_k remaining features.
    """

    def __init__(self, top_k=15, correlation_threshold=0.90):
        self.top_k = top_k
        self.correlation_threshold = correlation_threshold

    @staticmethod
    def _fisher_scores(frame, target):
        scores = {}
        target = np.asarray(target)

        for column in frame.columns:
            overall_mean = frame[column].mean()
            between_class = 0.0
            within_class = 0.0

            for class_value in np.unique(target):
                values = frame.loc[target == class_value, column]
                class_mean = values.mean()
                class_variance = values.var(ddof=0)

                between_class += len(values) * (class_mean - overall_mean) ** 2
                within_class += len(values) * class_variance

            scores[column] = between_class / (within_class + 1e-12)

        return pd.Series(scores).sort_values(ascending=False)

    def fit(self, X, y):
        frame = pd.DataFrame(X).copy()
        self.input_features_ = np.asarray(frame.columns, dtype=object)

        # Variance filtering: remove only constant columns.
        non_constant = frame.columns[frame.var(axis=0, ddof=0) > 0]
        frame = frame.loc[:, non_constant]

        if frame.shape[1] == 0:
            raise ValueError("All features have zero variance.")

        self.fisher_scores_ = self._fisher_scores(frame, y)

        # Greedy correlation pruning in descending Fisher-score order.
        corr = frame.corr().abs()
        retained = []

        for feature in self.fisher_scores_.index:
            if all(corr.loc[feature, kept] <= self.correlation_threshold
                   for kept in retained):
                retained.append(feature)

        self.correlation_retained_features_ = retained
        self.selected_features_ = retained[:min(self.top_k, len(retained))]

        if len(self.selected_features_) == 0:
            raise ValueError("No features remain after feature selection.")

        return self

    def transform(self, X):
        frame = pd.DataFrame(X).copy()
        return frame.loc[:, self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_features_, dtype=object)

Overwriting feature_selector.py


In [44]:
from feature_selector import StatisticalFeatureSelector

# Preview: fit the selector once on the training split to see what it picks.
# During model evaluation (Step 7) it is re-fitted inside every cross-validation fold.
selector_preview = StatisticalFeatureSelector(
    top_k=15,
    correlation_threshold=0.90
)
selector_preview.fit(X_train, y_train)

selected_features = list(selector_preview.selected_features_)
fisher_scores = selector_preview.fisher_scores_

print("Original features:", X_train.shape[1])
print("After constant-feature filtering:",
      int((X_train.var(axis=0, ddof=0) > 0).sum()))
print("After correlation pruning:",
      len(selector_preview.correlation_retained_features_))
print("Final selected features:", len(selected_features))

display(pd.DataFrame({
    "Selected Feature": selected_features,
    "Fisher Score": [fisher_scores[f] for f in selected_features]
}))

Original features: 30
After constant-feature filtering: 30
After correlation pruning: 21
Final selected features: 15


,Selected Feature,Fisher Score
0,worst concave points,1.755460
1,worst perimeter,1.588518
2,mean concavity,0.919150
3,worst concavity,0.807389
4,mean compactness,0.594848
5,worst compactness,0.569373
6,area error,0.474302
7,worst texture,0.255004
8,worst symmetry,0.227390
9,worst smoothness,0.225845


Visualize the final selected features from the training-set preview. The selector is fitted again inside each cross-validation fold during model evaluation, so validation folds never influence feature selection.

In [45]:
plt.figure(figsize=(10, 6))
sns.barplot(
    x=[fisher_scores[f] for f in selected_features],
    y=selected_features
)
plt.title("Selected Features Ranked by Fisher Score")
plt.xlabel("Fisher Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# **Step 7. Compare three classification algorithms**

We compare exactly three classifiers:

1. Logistic Regression
2. Decision Tree
3. K-Nearest Neighbors (KNN)


**Methodology (kept simple)**
- Every model is a Scikit-learn `Pipeline`, so feature selection and scaling are re-fitted on the training part of every fold (no data leakage).
- Cross-validation is **repeated** (5 folds x 3 repeats) so results are more stable on this small dataset.
- A small hyperparameter search (`GridSearchCV`) tunes one setting per classifier. For the selected-feature pipeline it also tunes how many features to keep (10, 15 or 20).
- The all-30-features baseline gets the same classifier tuning, so the before/after comparison is fair.
- Models are chosen using training cross-validation only. The test set stays untouched until Step 8.
- Malignant (class 0) is treated as the positive class for precision, recall and F1.

In [46]:
# Three classifiers only:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=3000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# One small, simple grid per classifier.
param_grids = {
    "Logistic Regression": {"classifier__C": [0.1, 1, 10]},
    "Decision Tree": {"classifier__max_depth": [3, 5, 7, None]},
    "K-Nearest Neighbors": {"classifier__n_neighbors": [3, 5, 7, 11]}
}

# Malignant (label 0) is the positive class.
scoring = {
    "accuracy": "accuracy",
    "recall_malignant": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_malignant": make_scorer(f1_score, pos_label=0, zero_division=0),
    "roc_auc": "roc_auc"
}

# 5 folds x 3 repeats = 15 validation scores per setting.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)


def run_search(pipeline, grid):
    search = GridSearchCV(
        pipeline, grid,
        scoring=scoring, refit="f1_malignant",
        cv=cv, n_jobs=1
    )
    search.fit(X_train, y_train)
    return search

searches_all = {}
searches_selected = {}
rows_before = []
rows_after = []


def summarize(model_name, search):
    """Pull the best setting's repeated-CV scores out of a GridSearchCV."""
    i = search.best_index_
    r = search.cv_results_
    return {
        "Model": model_name,
        "CV Accuracy": r["mean_test_accuracy"][i],
        "CV Recall (Malignant)": r["mean_test_recall_malignant"][i],
        "CV F1 (Malignant)": r["mean_test_f1_malignant"][i],
        "CV F1 Std": r["std_test_f1_malignant"][i],
        "CV ROC-AUC": r["mean_test_roc_auc"][i],
        "Best Settings": str(search.best_params_)
    }


for model_name, classifier in classifiers.items():

    # clone() gives each pipeline its own classifier, so they never share state.
    pipeline_all = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", clone(classifier))
    ])

    pipeline_selected = Pipeline([
        ("feature_selection", StatisticalFeatureSelector(correlation_threshold=0.90)),
        ("scaler", StandardScaler()),
        ("classifier", clone(classifier))
    ])

    grid_all = param_grids[model_name]
    grid_selected = {**grid_all, "feature_selection__top_k": [10, 15, 20]}

    searches_all[model_name] = run_search(pipeline_all, grid_all)
    searches_selected[model_name] = run_search(pipeline_selected, grid_selected)

    rows_before.append(summarize(model_name, searches_all[model_name]))
    rows_after.append(summarize(model_name, searches_selected[model_name]))

cv_before = pd.DataFrame(rows_before)
cv_after = pd.DataFrame(rows_after)

print("Cross-validation results using all 30 features (training data only)")
display(cv_before.round(4))
print("\n\n")
print("Cross-validation results using statistical feature selection (training data only)")
display(cv_after.round(4))

# Compact side-by-side table, used for model selection (Step 8) and the dashboard.
cv_results = pd.DataFrame({
    "Model": cv_before["Model"],
    "CV F1 Before (Malignant)": cv_before["CV F1 (Malignant)"],
    "CV F1 After (Malignant)": cv_after["CV F1 (Malignant)"],
    "CV Accuracy Before": cv_before["CV Accuracy"],
    "CV Accuracy After": cv_after["CV Accuracy"]
})

Cross-validation results using all 30 features (training data only)


,Model,CV Accuracy,CV Recall (Malignant),CV F1 (Malignant),CV F1 Std,CV ROC-AUC,Best Settings
0,Logistic Regression,0.9795,0.9510,0.9717,0.0148,0.9956,{'classifier__C': 0.1}
1,Decision Tree,0.9209,0.8941,0.8939,0.0320,0.9333,{'classifier__max_depth': 3}
2,K-Nearest Neighbors,0.9663,0.9353,0.9540,0.0229,0.9878,{'classifier__n_neighbors': 7}





Cross-validation results using statistical feature selection (training data only)


,Model,CV Accuracy,CV Recall (Malignant),CV F1 (Malignant),CV F1 Std,CV ROC-AUC,Best Settings
0,Logistic Regression,0.9714,0.9412,0.9607,0.0193,0.9957,"{'classifier__C': 1, 'feature_selection__top_k..."
1,Decision Tree,0.9297,0.8961,0.9045,0.0375,0.9350,"{'classifier__max_depth': 3, 'feature_selectio..."
2,K-Nearest Neighbors,0.9553,0.9314,0.9396,0.0184,0.9774,"{'classifier__n_neighbors': 3, 'feature_select..."


## **Step 8. Final evaluation on the untouched test set**

`GridSearchCV` has already re-fitted the best pipeline of each model on the complete training split. Each one is now evaluated once on the held-out test split. The best selected-feature model is chosen using the mean cross-validation F1-score for the malignant class, not test accuracy.

In [47]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Malignant)": precision_score(
            y_true, y_pred, pos_label=0, zero_division=0
        ),
        "Recall (Malignant)": recall_score(
            y_true, y_pred, pos_label=0, zero_division=0
        ),
        "F1 (Malignant)": f1_score(
            y_true, y_pred, pos_label=0, zero_division=0
        )
    }


results_before_rows = []
results_after_rows = []
fitted_models_all = {}
fitted_models_selected = {}

for model_name in classifiers:

    # Best pipeline using all 30 features (already fitted on the training split).
    model_all = searches_all[model_name].best_estimator_
    results_before_rows.append({
        "Model": model_name,
        **calculate_metrics(y_test, model_all.predict(X_test))
    })
    fitted_models_all[model_name] = model_all

    # Best pipeline using statistically selected features.
    model_selected = searches_selected[model_name].best_estimator_
    results_after_rows.append({
        "Model": model_name,
        **calculate_metrics(y_test, model_selected.predict(X_test))
    })
    fitted_models_selected[model_name] = model_selected

results_before = pd.DataFrame(results_before_rows)
results_after = pd.DataFrame(results_after_rows)

print("Test-set results using all 30 features")
display(results_before.round(4))
print("\n\n")
print("Test-set results using statistical feature selection")
display(results_after.round(4))

comparison = results_before.merge(
    results_after,
    on="Model",
    suffixes=(" Before", " After")
)



# Select the model using training-only cross-validation F1 after feature selection.
best_model_name = (
    cv_results.sort_values(
        "CV F1 After (Malignant)",
        ascending=False
    ).iloc[0]["Model"]
)

best_model = fitted_models_selected[best_model_name]
best_predictions = best_model.predict(X_test)

# The tuned model may keep a different number of features than the preview,
# so refresh the selected-feature list and scores from the final model.
final_selector = best_model.named_steps["feature_selection"]
selected_features = list(final_selector.selected_features_)
fisher_scores = final_selector.fisher_scores_
print()
print("\nModel selected using training CV:", best_model_name)
print("Best settings:", searches_selected[best_model_name].best_params_)
print("Number of features used:", len(selected_features))
print("Final held-out test metrics:")
display(pd.DataFrame([calculate_metrics(
    y_test, best_predictions
)]).round(4))

Test-set results using all 30 features


,Model,Accuracy,Precision (Malignant),Recall (Malignant),F1 (Malignant)
0,Logistic Regression,0.9737,0.9756,0.9524,0.9639
1,Decision Tree,0.9386,0.9268,0.9048,0.9157
2,K-Nearest Neighbors,0.9737,1.0000,0.9286,0.9630





Test-set results using statistical feature selection


,Model,Accuracy,Precision (Malignant),Recall (Malignant),F1 (Malignant)
0,Logistic Regression,0.9825,1.0000,0.9524,0.9756
1,Decision Tree,0.8947,0.8409,0.8810,0.8605
2,K-Nearest Neighbors,0.9561,0.9302,0.9524,0.9412




Model selected using training CV: Logistic Regression
Best settings: {'classifier__C': 1, 'feature_selection__top_k': 15}
Number of features used: 15
Final held-out test metrics:


,Accuracy,Precision (Malignant),Recall (Malignant),F1 (Malignant)
0,0.9825,1.0,0.9524,0.9756


## **Step 9. Visualize model comparison and confusion matrix**

The confusion matrix uses the class labels in the dataset: malignant (0) and benign (1). Pay particular attention to false negatives: malignant cases predicted as benign.

In [48]:
# Accuracy comparison for the three classifiers.
accuracy_comparison = pd.DataFrame({
    "Model": results_before["Model"],
    "Before Selection": results_before["Accuracy"] * 100,
    "After Selection": results_after["Accuracy"] * 100
}).set_index("Model")

ax = accuracy_comparison.plot(
    kind="bar",
    figsize=(10, 5),
    rot=0
)
ax.set_title("Test Accuracy: All Features vs. Selected Features")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.legend(title="Feature set")
plt.tight_layout()
plt.show()

# Confusion matrix for the CV-selected model.
cm = confusion_matrix(
    y_test,
    best_predictions,
    labels=[0, 1]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Malignant", "Benign"],
    yticklabels=["Malignant", "Benign"]
)
plt.title(f"Confusion Matrix – {best_model_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()

print("False negatives (malignant predicted as benign):", cm[0, 1])

print("Classification report for the selected model:")
print(classification_report(
    y_test,
    best_predictions,
    labels=[0, 1],
    target_names=["Malignant", "Benign"],
    zero_division=0
))

False negatives (malignant predicted as benign): 2
Classification report for the selected model:
              precision    recall  f1-score   support

   Malignant       1.00      0.95      0.98        42
      Benign       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



# **Step 10. Build a professional Gradio dashboard**

The dashboard contains six sections: Overview, Statistical Analysis, Feature Selection, Model Evaluation, Comparison and Prediction. It displays results produced by the notebook above rather than illustrative or hard-coded performance values.

The Prediction section supports both:
- **Individual patient input:** enter all 30 measurements in a form.
- **CSV upload:** predict one or more samples using a CSV with the original 30 feature columns.

**Responsible-use note:** this is an educational machine-learning demonstration, not a medical diagnostic tool. It must not be used to make clinical decisions.

In [49]:
# Dashboard plotting functions
def dashboard_variance_plot():
    fig, ax = plt.subplots(figsize=(9, 5))
    variance_df.head(12).sort_values("Raw Variance").plot(
        x="Feature", y="Raw Variance", kind="barh",
        ax=ax, legend=False, color="#6478D3"
    )
    ax.set_title("Top Features by Raw Variance")
    ax.set_xlabel("Raw variance (scale-dependent)")
    ax.set_ylabel("")
    fig.tight_layout()
    plt.close(fig)
    return fig


def dashboard_covariance_plot():
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        covariance_matrix, cmap="vlag", center=0,
        ax=ax, cbar_kws={"shrink": 0.75}
    )
    ax.set_title("Covariance Matrix")
    fig.tight_layout()
    plt.close(fig)
    return fig


def dashboard_correlation_plot():
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        correlation_matrix, cmap="coolwarm", center=0,
        vmin=-1, vmax=1, ax=ax,
        cbar_kws={"shrink": 0.75}
    )
    ax.set_title("Correlation Heatmap")
    fig.tight_layout()
    plt.close(fig)
    return fig


def dashboard_fisher_plot():
    selected_scores = fisher_scores.loc[selected_features].sort_values()
    fig, ax = plt.subplots(figsize=(9, 5))
    selected_scores.plot(kind="barh", ax=ax, color="#16A085")
    ax.set_title("Selected Features Ranked by Fisher Score")
    ax.set_xlabel("Fisher score")
    ax.set_ylabel("")
    fig.tight_layout()
    plt.close(fig)
    return fig


def dashboard_accuracy_plot():
    fig, ax = plt.subplots(figsize=(9, 5))
    accuracy_comparison.plot(kind="bar", ax=ax, color=["#9AA7B8", "#5367C5"])
    ax.set_title("Test Accuracy Before and After Feature Selection")
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Feature set")
    fig.tight_layout()
    plt.close(fig)
    return fig


def dashboard_confusion_plot():
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Malignant", "Benign"],
        yticklabels=["Malignant", "Benign"],
        ax=ax
    )
    ax.set_title(f"Confusion Matrix – {best_model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    fig.tight_layout()
    plt.close(fig)
    return fig


# Create a concise feature-ranking table.
feature_ranking_table = pd.DataFrame({
    "Feature": selected_features,
    "Fisher Score": [fisher_scores[f] for f in selected_features]
})

# Create a compact model comparison table for the UI.
ui_comparison = comparison.copy()
for column in ui_comparison.columns:
    if column != "Model":
        ui_comparison[column] = (ui_comparison[column] * 100).round(2)

print("Dashboard components prepared.")

Dashboard components prepared.


## **Step 11. Create the interactive Gradio application**

The dashboard uses a clean indigo-and-slate theme, summary cards, charts, model comparison tables and an individual-input prediction form.

In [50]:
dashboard_css = """
.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
}
#project-header {
    padding: 26px;
    border-radius: 18px;
    background: linear-gradient(120deg, #202B55, #5367C5);
    color: white;
    margin-bottom: 16px;
}
#project-header h1, #project-header p { color: white !important; }
.metric-card {
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 14px;
    background: var(--background-fill-secondary);
}
.section-note {
    color: var(--body-text-color-subdued);
    font-size: 0.92rem;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=dashboard_css,
    title="Statistical Feature Selection Dashboard"
) as app:

    gr.HTML("""
    <div id="project-header">
      <h1>Statistical Feature Selection for Classification</h1>
      <p>Breast Cancer Wisconsin Diagnostic Dataset · Statistical analysis ·
      Feature selection · Classification</p>
    </div>
    """)

    with gr.Tabs():

        # TAB 1: OVERVIEW
        with gr.Tab("Overview"):
            gr.Markdown("## Project overview")
            gr.Markdown(
                "This application investigates how variance, covariance, "
                "correlation and Fisher score can be used to select useful "
                "features before classification."
            )
            with gr.Row():
                gr.Number(value=len(X), label="Total samples", interactive=False)
                gr.Number(value=X.shape[1], label="Original features", interactive=False)
                gr.Number(value=len(selected_features), label="Selected features", interactive=False)
                gr.Number(value=2, label="Target classes", interactive=False)

            with gr.Row():
                gr.Markdown(f"""
                ### Dataset
                **Breast Cancer Wisconsin (Diagnostic)**

                - {len(X)} samples and {X.shape[1]} numerical features
                - {int((y == 1).sum())} benign and {int((y == 0).sum())} malignant cases
                - Target: malignant (0), benign (1)
                """)
                gr.Markdown("""
                ### Evaluation setup
                - Stratified 80:20 train/test split
                - Repeated 5-fold cross-validation (3 repeats) with a small hyperparameter search, on training data only
                - Three classifiers: Logistic Regression, Decision Tree and KNN
                - Best model chosen using cross-validation malignant-class F1
                """)

            gr.Markdown(
                "<div class='section-note'>Educational demonstration only; "
                "not a medical diagnostic system.</div>"
            )

        # TAB 2: STATISTICAL ANALYSIS
        with gr.Tab("Statistical Analysis"):
            gr.Markdown("## Statistical analysis")
            gr.Markdown(
                "Raw variance depends on feature measurement scale. "
                "The final pipeline uses variance filtering only to remove "
                "constant features. Correlation is scale-independent."
            )
            with gr.Tab("Variance"):
                gr.Plot(value=dashboard_variance_plot)
            with gr.Tab("Covariance"):
                gr.Plot(value=dashboard_covariance_plot)
            with gr.Tab("Correlation"):
                gr.Plot(value=dashboard_correlation_plot)
                gr.Dataframe(
                    value=high_correlation.round(3),
                    label="Highly correlated feature pairs (|r| > 0.90)"
                )

        # TAB 3: FEATURE SELECTION
        with gr.Tab("Feature Selection"):
            gr.Markdown("## Selected features")
            gr.Markdown(
                "Features are ranked by Fisher score. Correlated pairs are "
                "pruned by retaining the higher-scoring feature, then the "
                "top features are selected (how many is tuned in cross-validation)."
            )
            gr.Number(
                value=len(selected_features),
                label="Selected feature count",
                interactive=False
            )
            gr.Dataframe(
                value=feature_ranking_table.round(5),
                label="Selected feature ranking"
            )
            gr.Plot(value=dashboard_fisher_plot)

        # TAB 4: MODEL EVALUATION
        with gr.Tab("Model Evaluation"):
            gr.Markdown("## Final test-set results")
            gr.Markdown(
                "Precision, recall and F1-score below treat malignant (class 0) "
                "as the positive class."
            )
            gr.Dataframe(
                value=results_after.round(4),
                label="Three classifiers using selected features"
            )
            gr.Markdown(f"### Model selected by training cross-validation: {best_model_name}")
            gr.Dataframe(
                value=pd.DataFrame([calculate_metrics(y_test, best_predictions)]).round(4),
                label="Held-out test metrics"
            )
            gr.Plot(value=dashboard_confusion_plot)

        # TAB 5: COMPARISON
        with gr.Tab("Comparison"):
            gr.Markdown("## Before vs. after feature selection")
            gr.Markdown(
                "All metrics are measured on the same untouched test set. "
                "Values are percentages."
            )
            gr.Dataframe(
                value=ui_comparison.round(2),
                label="Three-classifier performance comparison (%)"
            )
            gr.Plot(value=dashboard_accuracy_plot)
            gr.Markdown("### Cross-validation model-selection results")
            gr.Dataframe(
                value=cv_results.round(4),
                label="Repeated 5-fold training cross-validation"
            )

        # TAB 6: PREDICTION
        with gr.Tab("Prediction"):
            gr.Markdown("## Individual patient measurement prediction")
            gr.Markdown(
                "Enter all 30 measurements below. The initial values are "
                "dataset medians for demonstration; replace them with the "
                "measurements you want to test."
            )

            patient_inputs = []
            feature_names = list(X.columns)

            for start in range(0, len(feature_names), 5):
                with gr.Row():
                    for feature in feature_names[start:start + 5]:
                        component = gr.Number(
                            label=feature,
                            value=float(X[feature].median()),
                            precision=5
                        )
                        patient_inputs.append(component)

            individual_predict_button = gr.Button(
                "Predict individual sample",
                variant="primary"
            )
            individual_prediction_output = gr.Markdown()

            def predict_individual(*values):
                if len(values) != len(feature_names):
                    raise gr.Error("Please provide all 30 feature values.")

                if any(value is None or not np.isfinite(float(value))
                       for value in values):
                    raise gr.Error("All 30 measurements must be valid numbers.")

                patient = pd.DataFrame(
                    [list(map(float, values))],
                    columns=feature_names
                )

                predicted_class = int(best_model.predict(patient)[0])
                probabilities = best_model.predict_proba(patient)[0]
                confidence = float(probabilities[predicted_class] * 100)

                label = data.target_names[predicted_class].capitalize()
                model_text = best_model_name + (
                    " (its probabilities are coarse and not true confidence)"
                    if best_model_name in ["Decision Tree", "K-Nearest Neighbors"] else ""
                )
                color = "#D97706" if predicted_class == 0 else "#15803D"

                return (
                    f"### Prediction: <span style='color:{color}'>{label}</span>\n\n"
                    f"**Model:** {model_text}  \n"
                    f"**Model probability for predicted class:** {confidence:.2f}%  \n\n"
                    "**Important:** This is a model output for an educational "
                    "dataset exercise, not a clinical diagnosis. Consult a "
                    "qualified healthcare professional for medical decisions."
                )

            individual_predict_button.click(
                fn=predict_individual,
                inputs=patient_inputs,
                outputs=individual_prediction_output
            )

            gr.Markdown("---")
            gr.Markdown("## Batch prediction using CSV")
            gr.Markdown(
                "Optional: upload a CSV containing all 30 original feature "
                "columns. Column names must match the dataset."
            )
            csv_input = gr.File(
                label="Upload patient measurements CSV",
                file_types=[".csv"],
                type="filepath"
            )
            csv_predict_button = gr.Button(
                "Predict uploaded samples",
                variant="secondary"
            )
            csv_prediction_output = gr.Dataframe(
                label="Batch prediction results"
            )

            def predict_csv(file_path):
                if file_path is None:
                    raise gr.Error("Please upload a CSV file.")

                patient_data = pd.read_csv(file_path)
                missing_columns = [
                    column for column in feature_names
                    if column not in patient_data.columns
                ]

                if missing_columns:
                    raise gr.Error(
                        "CSV is missing required columns: " +
                        ", ".join(missing_columns)
                    )

                patient_data = patient_data[feature_names].copy()

                if patient_data.isnull().any().any():
                    raise gr.Error("The CSV contains missing values.")

                try:
                    patient_data = patient_data.astype(float)
                except (ValueError, TypeError):
                    raise gr.Error("All feature columns must contain numeric values.")

                if not np.isfinite(patient_data.to_numpy()).all():
                    raise gr.Error("The CSV contains invalid numeric values.")

                predictions = best_model.predict(patient_data)
                probabilities = best_model.predict_proba(patient_data)

                return pd.DataFrame({
                    "Sample": np.arange(1, len(patient_data) + 1),
                    "Prediction": [
                        data.target_names[int(value)].capitalize()
                        for value in predictions
                    ],
                    "Predicted-class probability (%)": [
                        round(float(probabilities[i, int(predictions[i])] * 100), 2)
                        for i in range(len(predictions))
                    ]
                })

            csv_predict_button.click(
                fn=predict_csv,
                inputs=csv_input,
                outputs=csv_prediction_output
            )

    gr.Markdown(
        "<center><small>Statistical Feature Selection for Classification · "
        "Breast Cancer Wisconsin Diagnostic Dataset · Educational use only</small></center>"
    )

print("Launching the professional Gradio dashboard...")
app.launch(share=True)

Launching the professional Gradio dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d5e775ff1ef0b7fcb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Step 12. Save the trained model and project outputs**

Save the best selected-feature pipeline and the feature list. The saved pipeline includes the feature selector, scaler and classifier, so the same preprocessing is applied during future predictions. The `StatisticalFeatureSelector` class lives in `feature_selector.py`, so keep that file next to the saved model.

In [51]:
import joblib
import json
import sklearn

joblib.dump(
    best_model,
    "breast_cancer_selected_feature_pipeline.joblib"
)

with open("selected_features.json", "w") as file:
    json.dump(selected_features, file, indent=4)

results_after.to_csv("classification_results_after_selection.csv", index=False)
comparison.to_csv("before_after_comparison.csv", index=False)
cv_results.to_csv("cross_validation_results.csv", index=False)

# Record library versions so the saved model can be reloaded safely.
with open("requirements.txt", "w") as file:
    file.write(f"scikit-learn=={sklearn.__version__}\n")
    file.write(f"pandas=={pd.__version__}\n")
    file.write(f"numpy=={np.__version__}\n")
    file.write("joblib\n")

print("Saved:")
print("- breast_cancer_selected_feature_pipeline.joblib")
print("- selected_features.json")
print("- classification_results_after_selection.csv")
print("- before_after_comparison.csv")
print("- cross_validation_results.csv")
print("- requirements.txt")
print("\nTo reload the model elsewhere, keep feature_selector.py in the same folder:")
print("  from feature_selector import StatisticalFeatureSelector")
print("  model = joblib.load('breast_cancer_selected_feature_pipeline.joblib')")

Saved:
- breast_cancer_selected_feature_pipeline.joblib
- selected_features.json
- classification_results_after_selection.csv
- before_after_comparison.csv
- cross_validation_results.csv
- requirements.txt

To reload the model elsewhere, keep feature_selector.py in the same folder:
  from feature_selector import StatisticalFeatureSelector
  model = joblib.load('breast_cancer_selected_feature_pipeline.joblib')


# **Final project notes**

- Exactly three classifiers are compared: Logistic Regression, Decision Tree and KNN.
- Random Forest and SVM have been removed.
- Cross-validation is repeated (5 folds x 3 repeats) and used only on the training data.
- Feature selection and scaling are refitted inside every cross-validation fold.
- A small hyperparameter search tunes each classifier and the number of selected features.
- The held-out test set is used only for final evaluation; it is small (114 samples), so tiny differences may be noise.
- The best model is selected using cross-validation F1-score for the malignant class.
- The individual prediction form accepts all 30 original feature measurements.
- This notebook is an educational project and must not be presented as a clinically validated diagnostic system.

**Before submitting:** Run all cells from top to bottom in Google Colab. The Gradio public URL is temporary and may change each time the final dashboard cell is run.